# Credit Risk Scoring — XGBoost, MLflow, FastAPI

Credit risk assessment is one of the oldest and most consequential applications of machine learning in finance. A model that approves a bad loan exposes the lender to potential loss of the entire principal. A model that rejects a good loan loses a customer and revenue. These errors have asymmetric costs — in the German Credit dataset, the standard assumption is that a bad loan costs 5 times more than a missed good loan. This asymmetry must be built into both the model training and the evaluation metric.

This notebook walks through the complete pipeline: data exploration, preprocessing, model training across three classifiers tracked with MLflow, cost-weighted evaluation, SHAP explainability, and FastAPI deployment.

In [ ]:
import json
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import mlflow
import mlflow.sklearn

from pathlib import Path
from sklearn.datasets import fetch_openml
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix,
)
from xgboost import XGBClassifier

from config import (
    MODEL_DIR, PLOTS_DIR, MLFLOW_TRACKING_URI, MLFLOW_EXPERIMENT_NAME,
    TARGET_COLUMN, POSITIVE_CLASS, CLASS_WEIGHT, RANDOM_STATE, TEST_SIZE,
    RISK_BANDS,
)

print('Config constants:')
print(f'  MLFLOW_TRACKING_URI = {MLFLOW_TRACKING_URI}')
print(f'  MLFLOW_EXPERIMENT_NAME = {MLFLOW_EXPERIMENT_NAME}')
print(f'  TARGET_COLUMN = {TARGET_COLUMN}')
print(f'  POSITIVE_CLASS = {POSITIVE_CLASS}')
print(f'  CLASS_WEIGHT = {CLASS_WEIGHT}')
print(f'  RANDOM_STATE = {RANDOM_STATE}')
print(f'  TEST_SIZE = {TEST_SIZE}')

## German Credit Dataset

1,000 loan applicants, 20 features. Target: `class` — `good` (700 applicants) or `bad` (300 applicants).

The **70/30 class imbalance** is significant for evaluation. A model that approves every application achieves 70% accuracy while correctly identifying **zero bad loans**. Accuracy is therefore a misleading metric for this problem.

Key features:
- `checking_status` — current account balance (one of the strongest predictors)
- `duration` — loan term in months
- `credit_history` — past repayment behaviour
- `credit_amount` — loan size in Deutschmarks
- `savings_status` — savings account balance
- `employment` — duration at current job
- `age` — applicant age

In [ ]:
data = fetch_openml('credit-g', version=1, as_frame=True)
df = data.frame

print(f'Shape: {df.shape}')
print(f'\nClass distribution:')
class_counts = df[TARGET_COLUMN].value_counts()
print(class_counts)
print(f'\nImbalance ratio (bad/good): {class_counts["bad"] / class_counts["good"]:.2f}')
print(f'\nFeature types:')
print(df.dtypes)

In [ ]:
PLOTS_DIR.mkdir(exist_ok=True)

df['target'] = (df[TARGET_COLUMN] == POSITIVE_CLASS).astype(int)
df_eda = df.drop(columns=[TARGET_COLUMN])

numeric_features = df_eda.select_dtypes(include=['number']).columns.tolist()
numeric_features = [f for f in numeric_features if f != 'target']
categorical_features = df_eda.select_dtypes(include=['object', 'category']).columns.tolist()

# 01 — class distribution
fig, ax = plt.subplots(figsize=(7, 5))
labels = ['Good (0)', 'Bad (1)']
values = [class_counts['good'], class_counts['bad']]
colours = ['#27AE60', '#C0392B']
bars = ax.bar(labels, values, color=colours, edgecolor='white', linewidth=1.5)
for bar, val in zip(bars, values):
    pct = val / sum(values) * 100
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 8,
            f'{val} ({pct:.0f}%)', ha='center', fontweight='bold')
ax.set_title('Class Distribution', fontsize=14, fontweight='bold')
ax.set_ylabel('Count')
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig(PLOTS_DIR / '01_class_distribution.png', dpi=150)
plt.show()

# 02-04 — KDE plots
for i, feature in enumerate(['credit_amount', 'duration', 'age'], start=2):
    fig, ax = plt.subplots(figsize=(8, 4))
    for label, colour, val in [('Good', '#27AE60', 0), ('Bad', '#C0392B', 1)]:
        subset = df_eda[df_eda['target'] == val][feature]
        subset.plot.kde(ax=ax, label=label, color=colour, linewidth=2)
        ax.axvline(subset.median(), color=colour, linestyle='--', alpha=0.6)
    ax.set_title(f'{feature.replace("_", " ").title()} by Risk Class', fontsize=12, fontweight='bold')
    ax.legend()
    ax.spines[['top', 'right']].set_visible(False)
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / f'0{i}_{feature}_by_risk.png', dpi=150)
    plt.show()

# 05 — correlation heatmap
fig, ax = plt.subplots(figsize=(9, 7))
corr = df_eda[numeric_features + ['target']].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn_r',
            center=0, square=True, ax=ax)
ax.set_title('Feature Correlation', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(PLOTS_DIR / '05_feature_correlation.png', dpi=150)
plt.show()

# 06 — categorical default rates
cat_features_to_plot = ['checking_status', 'credit_history', 'purpose',
                         'savings_status', 'employment', 'personal_status']
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for ax, feature in zip(axes.flatten(), cat_features_to_plot):
    default_rate = df_eda.groupby(feature)['target'].mean().sort_values(ascending=False)
    colours = ['#C0392B' if r > 0.4 else '#F39C12' if r > 0.25 else '#27AE60'
               for r in default_rate.values]
    default_rate.plot.bar(ax=ax, color=colours, edgecolor='white')
    ax.set_title(feature.replace('_', ' ').title(), fontweight='bold')
    ax.set_ylabel('Default Rate')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=35, ha='right', fontsize=8)
    ax.set_ylim(0, 1)
    ax.axhline(0.3, color='grey', linestyle='--', alpha=0.5)
    ax.spines[['top', 'right']].set_visible(False)
fig.suptitle('Default Rate by Categorical Feature', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig(PLOTS_DIR / '06_categorical_features.png', dpi=150, bbox_inches='tight')
plt.show()

## Preprocessing

**ColumnTransformer** applies different transformations to numeric and categorical columns in one step:
- **StandardScaler** for numeric features — centres and scales to unit variance. Required for Logistic Regression (gradient descent converges faster with normalised features) and has no negative effect on tree-based models.
- **OneHotEncoder(drop='first')** for categorical features — encodes each category as a binary column. `drop='first'` removes the first category of each feature to avoid multicollinearity (the dropped category is the reference level). `sparse_output=False` returns a dense array compatible with SHAP.

**Pipeline** wraps the preprocessor and classifier together. This solves a critical problem: the scaler must be **fit only on training data** and then **applied** to test data. Fitting on the full dataset (including test) leaks information from the test set into the model, artificially inflating evaluation metrics. With Pipeline, calling `pipeline.fit(X_train, y_train)` fits both the preprocessor and classifier on training data only. Calling `pipeline.predict(X_test)` applies the fitted preprocessor before prediction — no leakage.

The saved `best_model.pkl` includes the full pipeline, so the API accepts raw input and produces predictions without a separate preprocessing step.

In [ ]:
X = df_eda.drop(columns=['target'])
y = df_eda['target']

preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), numeric_features),
    ('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'),
     categorical_features),
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE,
)

print(f'Train: {X_train.shape[0]} samples — {y_train.sum()} bad ({y_train.mean():.1%})')
print(f'Test:  {X_test.shape[0]} samples — {y_test.sum()} bad ({y_test.mean():.1%})')

## MLflow Experiment Tracking

MLflow tracks every training run with its hyperparameters, metrics, and model artefacts. This means I can compare all three models in the MLflow UI, reproduce any run exactly, and register the best model for deployment — all from the same experiment.

Previous projects in this portfolio used a manual `model_registry.csv` to track runs. MLflow is the production implementation of that pattern: structured, queryable, and integrated with model serving.

**MLflow UI screenshot** — *add after running*

Each model is logged with:
- Hyperparameters (via `mlflow.log_params`)
- Six metrics including `cost_weighted_score` (via `mlflow.log_metric`)
- The full sklearn Pipeline artefact (via `mlflow.sklearn.log_model`)
- EDA plots as artefacts

In [ ]:
def cost_weighted_score_metric(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    fn = cm[1][0]
    fp = cm[0][1]
    total_bad = y_true.sum()
    total_good = len(y_true) - total_bad
    return 1 - (5 * fn + fp) / (5 * total_bad + total_good)


def evaluate_model(pipeline, X_test, y_test):
    y_pred = pipeline.predict(X_test)
    y_prob = pipeline.predict_proba(X_test)[:, 1]
    return {
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred),
        'recall': recall_score(y_test, y_pred),
        'f1': f1_score(y_test, y_pred),
        'roc_auc': roc_auc_score(y_test, y_prob),
        'cost_weighted_score': cost_weighted_score_metric(y_test, y_pred),
    }


mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)

models_config = [
    ('LogisticRegression', LogisticRegression(
        random_state=RANDOM_STATE, max_iter=1000, class_weight=CLASS_WEIGHT)),
    ('RandomForest', RandomForestClassifier(
        n_estimators=100, random_state=RANDOM_STATE, class_weight=CLASS_WEIGHT)),
    ('XGBoost', XGBClassifier(
        n_estimators=100, random_state=RANDOM_STATE,
        scale_pos_weight=5, eval_metric='logloss', verbosity=0)),
]

all_metrics = {}
all_pipelines = {}
run_ids = {}

for model_name, model in models_config:
    pipeline = Pipeline([('preprocessor', preprocessor), ('classifier', model)])
    with mlflow.start_run(run_name=model_name) as run:
        run_ids[model_name] = run.info.run_id
        pipeline.fit(X_train, y_train)
        metrics = evaluate_model(pipeline, X_test, y_test)
        mlflow.log_params({'model': model_name})
        for k, v in metrics.items():
            mlflow.log_metric(k, v)
        mlflow.sklearn.log_model(pipeline, 'model')
    all_metrics[model_name] = metrics
    all_pipelines[model_name] = pipeline
    print(f'{model_name}: cost_weighted={metrics["cost_weighted_score"]:.4f}  roc_auc={metrics["roc_auc"]:.4f}')

print('\nComparison table:')
pd.DataFrame(all_metrics).T.round(4)

## Cost-Weighted Evaluation

ROC-AUC measures discrimination ability — how well the model ranks positive above negative cases. For credit risk, it does not directly measure business cost.

The **cost-weighted score** penalises false negatives (bad loans approved) 5× more than false positives (good loans rejected), reflecting the real asymmetry in lending decisions:

$$\text{cost\_weighted\_score} = 1 - \frac{5 \times FN + FP}{5 \times \text{total\_bad} + \text{total\_good}}$$

- **FN** — bad loans approved (dangerous: the bank loses the principal)
- **FP** — good loans rejected (costly but safe: the bank misses revenue)

A model with lower ROC-AUC but higher cost-weighted score is preferable for deployment in this context — it minimises the bank's expected loss.

In [ ]:
best_model_name = max(all_metrics, key=lambda n: all_metrics[n]['cost_weighted_score'])
best_pipeline = all_pipelines[best_model_name]
print(f'Winner: {best_model_name}')
print(f'Cost-weighted score: {all_metrics[best_model_name]["cost_weighted_score"]:.4f}')

# 10 — model comparison chart
metric_names = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc', 'cost_weighted_score']
x = np.arange(len(metric_names))
width = 0.25
fig, ax = plt.subplots(figsize=(14, 6))
colours = ['#3498DB', '#E67E22', '#2ECC71']
for i, (name, m) in enumerate(all_metrics.items()):
    values = [m[k] for k in metric_names]
    ax.bar(x + i * width, values, width, label=name, color=colours[i], edgecolor='white')
ax.set_xticks(x + width)
ax.set_xticklabels([m.replace('_', '\n') for m in metric_names], fontsize=9)
ax.set_ylim(0, 1.15)
ax.set_ylabel('Score')
ax.set_title('Model Comparison — All Metrics', fontsize=14, fontweight='bold')
ax.legend()
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig(PLOTS_DIR / '10_model_comparison.png', dpi=150)
plt.show()

## SHAP Explainability for Credit Decisions

Explainability is not optional in credit scoring — it is legally required in many jurisdictions. The EU AI Act and US Equal Credit Opportunity Act both require that adverse credit decisions can be explained to the applicant.

SHAP (SHapley Additive exPlanations) provides per-application explanations by attributing each feature's contribution to the model's output. A positive SHAP value means the feature pushed the prediction toward default (higher risk); a negative SHAP value means the feature pushed the prediction toward good credit.

`risk_explainer.py` translates raw SHAP values into statements a loan officer would recognise:

> *"Your application was declined primarily because of your loan duration (48 months), lack of checking account, and previous payment delays."*

This is the output the FastAPI `/score` endpoint returns for every prediction.

In [ ]:
X_test_transformed = best_pipeline.named_steps['preprocessor'].transform(X_test)
classifier = best_pipeline.named_steps['classifier']

ohe = best_pipeline.named_steps['preprocessor'].named_transformers_['cat']
cat_feature_names = ohe.get_feature_names_out(categorical_features).tolist()
all_feature_names = numeric_features + cat_feature_names

if best_model_name == 'LogisticRegression':
    explainer = shap.LinearExplainer(classifier, X_test_transformed)
else:
    explainer = shap.Explainer(classifier, X_test_transformed)

shap_values = explainer(X_test_transformed)
shap_vals_array = shap_values.values if hasattr(shap_values, 'values') else shap_values

# 07 — SHAP summary
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_vals_array, X_test_transformed,
                  feature_names=all_feature_names, show=False, max_display=15)
plt.title(f'SHAP Feature Importance — {best_model_name}', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(PLOTS_DIR / '07_shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()

y_prob_test = best_pipeline.predict_proba(X_test)[:, 1]
highest_risk_idx = np.argmax(y_prob_test)
lowest_risk_idx = np.argmin(y_prob_test)

# 08 — waterfall highest risk
plt.figure(figsize=(10, 6))
shap.waterfall_plot(shap_values[highest_risk_idx], max_display=10, show=False)
plt.title(f'Highest Risk Applicant — p={y_prob_test[highest_risk_idx]:.2f}', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(PLOTS_DIR / '08_shap_waterfall_highrisk.png', dpi=150, bbox_inches='tight')
plt.show()

# 09 — waterfall lowest risk
plt.figure(figsize=(10, 6))
shap.waterfall_plot(shap_values[lowest_risk_idx], max_display=10, show=False)
plt.title(f'Lowest Risk Applicant — p={y_prob_test[lowest_risk_idx]:.2f}', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(PLOTS_DIR / '09_shap_waterfall_lowrisk.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
from risk_explainer import shap_values_to_risk_factors

# Worked example: highest-risk applicant
shap_vals_example = shap_vals_array[highest_risk_idx]
risk_factors, protective_factors = shap_values_to_risk_factors(
    shap_vals_example.tolist(), all_feature_names, top_n=5,
)

print(f'Highest-risk applicant (p={y_prob_test[highest_risk_idx]:.2f})\n')
print('Top SHAP values:')
paired = sorted(zip(shap_vals_example, all_feature_names), key=lambda x: abs(x[0]), reverse=True)[:8]
print(f'{"Feature":<45} {"SHAP":>8}')
print('-' * 55)
for val, name in paired:
    print(f'{name:<45} {val:>8.4f}')

print('\nRisk factors:')
for f in risk_factors: print(f'  — {f}')
print('\nProtective factors:')
for f in protective_factors: print(f'  + {f}')

## FastAPI vs Flask for This Project

Previous projects in this portfolio used Flask. Credit risk scoring uses FastAPI for three reasons specific to this use case:

1. **API-first design.** A bank integrating this model calls `POST /score` programmatically — they don't use a form. FastAPI's auto-generated OpenAPI docs at `/docs` make the API immediately explorable without a separate documentation step.

2. **Input validation.** Pydantic models validate every field automatically and return structured 422 errors with field-level detail. Flask requires manual validation logic for the same behaviour — checking types, ranges, and allowed values for all ten input fields.

3. **Async support.** FastAPI is ASGI-native. The `/score/batch` endpoint can process 100 applications concurrently — Flask's WSGI architecture processes them sequentially.

The tradeoff: FastAPI requires `uvicorn` (or `gunicorn` with `UvicornWorker`) as the server — slightly more configuration than Flask's built-in development server. On Azure App Service, the startup command is:
```
gunicorn main:app --workers 1 --worker-class uvicorn.workers.UvicornWorker --bind 0.0.0.0:8000
```

## Docker Compose Architecture

Docker Compose runs two services locally:
- **api** — FastAPI scoring API on port 8000
- **mlflow** — MLflow tracking server on port 5000

This mirrors a production microservices architecture where the ML model server and the experiment tracking server are separate concerns. The `api` service has `depends_on: mlflow` — it won't start until MLflow is healthy.

For local development, `docker-compose up` replaces running both services manually in separate terminals. The `models/` directory is mounted as a volume so trained model files on the host are immediately available to the containerised API without rebuilding the image.

```bash
docker-compose up    # start both services
docker-compose down  # stop and remove containers
```

## Azure App Service Deployment

FastAPI is an ASGI framework — it requires an ASGI server to run. Uvicorn is the standard ASGI server. On Azure App Service, gunicorn acts as the process manager with UvicornWorker as the worker class, combining gunicorn's process management with uvicorn's ASGI handling.

The startup command is:
```
gunicorn main:app --workers 1 --worker-class uvicorn.workers.UvicornWorker --bind 0.0.0.0:8000 --timeout 600
```

Deployment sequence:
```bash
az group create --name credit-risk-rg --location westeurope
az appservice plan create --name credit-risk-plan --resource-group credit-risk-rg --sku B1 --is-linux
# Scale to F1 via portal after creation
az webapp create --name credit-risk-scorer --resource-group credit-risk-rg --plan credit-risk-plan --runtime "PYTHON:3.11"
az webapp config set --name credit-risk-scorer --resource-group credit-risk-rg --startup-file "gunicorn main:app --workers 1 --worker-class uvicorn.workers.UvicornWorker --bind 0.0.0.0:8000 --timeout 600"
az webapp config appsettings set --name credit-risk-scorer --resource-group credit-risk-rg --settings SCM_DO_BUILD_DURING_DEPLOYMENT=true
cd credit-risk-scorer && zip -r deploy.zip . -x "*.git*" -x "venv/*" -x "__pycache__/*" -x "*.ipynb_checkpoints*" -x "mlruns/*"
az webapp deployment source config-zip --name credit-risk-scorer --resource-group credit-risk-rg --src deploy.zip
```

## Key Findings

*TBD — populate after running train.py.*

Fill in:
- Winning model and its cost-weighted score vs baselines
- Top 3 SHAP features
- ROC-AUC comparison
- Precision/recall on the bad credit class

## Business Recommendations

Based on SHAP analysis, the three strongest predictors of default in this dataset are: *[TBD after training]*.

Recommended actions for the credit committee:

1. **[TBD — cite specific SHAP finding with number]** — e.g., if `checking_status_no checking` has the highest SHAP importance, recommend requiring applicants without checking accounts to provide additional collateral.

2. **[TBD]** — e.g., if `duration` is a top predictor, consider tighter thresholds for loans exceeding 36 months.

3. **[TBD]** — e.g., if `credit_history_delayed previously` is highly predictive, recommend a mandatory waiting period after late payments before loan approval.

The SHAP waterfall plots (charts 08 and 09) provide applicant-level explanations that can be included in adverse action notices, satisfying EU AI Act and ECOA explainability requirements.

## GitHub Actions CI

The CI pipeline runs `pytest tests/ -v` on every push to `main`. This catches regressions in the API endpoints — if a code change breaks `POST /score`, the push fails before it reaches Azure.

The pipeline does **not** run `train.py` (requires dataset download, several minutes) and does **not** deploy to Azure (manual deployment via `az` CLI).

For a production credit scoring system, the CI pipeline would also include:
- **Data validation checks** — ensure incoming data distributions match training data
- **Model performance regression tests** — compare a newly trained model against a minimum cost-weighted score threshold before allowing deployment
- **Automated deployment** — `az webapp deployment source config-zip` triggered on merge to main